# Annotated Field Catalog Generator

Generate an Excel catalog of legacy field names, definitions, source labels, calculated formulas, and target database field names.

In [ ]:
import pandas as pd
import openpyxl
from pathlib import Path

source_excel_path = Path('Coffee Grounds Data.xlsx')
output_catalog_path = Path('field_catalog.xlsx')

print('Source workbook:', source_excel_path)
print('Output catalog:', output_catalog_path)

## Load source definitions and metadata from Excel

Read the legacy workbook and inspect the `Collection DB` sheet for the raw field layout and any embedded formulas.

In [ ]:
wb = openpyxl.load_workbook(source_excel_path, data_only=False)
print('Workbook sheets:', wb.sheetnames)
sheet_name = 'Collection DB'
if sheet_name not in wb.sheetnames:
    raise ValueError(f'Sheet {sheet_name} not found in workbook')
sheet = wb[sheet_name]  # legacy raw field sheet
rows = list(sheet.iter_rows(values_only=False))
headers = [cell.value for cell in rows[0]]
print('Header labels:', headers)

sample_rows = []
for row in rows[1:11]:
    sample_rows.append([cell.value for cell in row[:12]])
print('Sample rows (first 10):')
for sample in sample_rows:
    print(sample)

## Parse formulas and identify calculated fields

Capture calculated source fields by detecting Excel formulas in the sheet and storing the expression text.

In [ ]:
legacy_fields = []
formula_columns = set()

for row in rows[1:]:
    if all(cell.value is None for cell in row):
        continue
    for idx, cell in enumerate(row):
        if isinstance(cell.value, str) and cell.value.startswith('='):
            formula_columns.add(idx)

for idx, label in enumerate(headers):
    if label is None:
        continue
    legacy_fields.append({
        'legacy_field_name': str(label).strip(),
        'excel_column_index': idx + 1,
        'calculated_field': idx in formula_columns,
        'formula_text': None,
    })

for row in rows[1:20]:
    for item in legacy_fields:
        idx = item['excel_column_index'] - 1
        if item['formula_text'] is None and idx < len(row):
            cell = row[idx]
            if isinstance(cell.value, str) and cell.value.startswith('='):
                item['formula_text'] = cell.value

legacy_df = pd.DataFrame(legacy_fields)
legacy_df.head(20)

## Map field origins to alteryx/tableau/excel labels

Assign a source label to each field based on the legacy worksheet and known derived metrics from Alteryx/Tableau.

In [ ]:
def infer_target_field_name(label):
    text = str(label).strip().lower()
    mapping = {
        'store number': 'store_number',
        'store name': 'raw_store_name',
        'pickup date': 'pickup_date',
        'week number': 'week_number',
        'scg mass (lbs)': 'scg_mass_lbs',
        'net lbs': 'net_lbs',
        'pickup initiated by': 'pickup_initiated_by',
        'master gardener': 'master_gardener',
        'mg deposit date': 'mg_deposit_date',
        'cardboard (lbs)': 'cardboard_lbs',
        'food waste (lbs)': 'food_waste_lbs',
        'route': 'route',
        'miles driven': 'miles_driven',
        'truck odometer': 'truck_odometer',
        'notes': 'notes',
        'raw days between collections': 'raw_days_between_collections',
        'raw days since first collection': 'raw_days_since_first_collection',
    }
    return mapping.get(text, None)

derived_fields = {
    'co2e_lbs': 'alteryx',
    'transportation_co2e': 'tableau',
    'scg_lbs_per_mile': 'tableau',
    'co2e_avoided_per_mile': 'tableau',
    'per_day_lbs': 'tableau',
    'rubicon_period': 'tableau',
    'year_and_week': 'tableau',
    'days_since_first_collection': 'tableau',
    'days_between_collections': 'tableau',
    'running_total': 'tableau',
}

catalog_rows = []
for item in legacy_fields:
    target = infer_target_field_name(item['legacy_field_name'])
    origin = 'excel'
    if item['calculated_field']:
        origin = 'excel'
    catalog_rows.append({
        'legacy_field_name': item['legacy_field_name'],
        'target_database_field_name': target or 'unknown',
        'origin_label': origin,
        'calculated_field': item['calculated_field'],
        'formula_text': item['formula_text'] or '',
        'definition': '',
    })

for field_name, origin in derived_fields.items():
    catalog_rows.append({
        'legacy_field_name': field_name,
        'target_database_field_name': field_name,
        'origin_label': origin,
        'calculated_field': True,
        'formula_text': '',
        'definition': '',
    })

catalog_df = pd.DataFrame(catalog_rows)
catalog_df.head(20)

## Assemble database field definitions DataFrame

Build the final annotated catalog data frame, including target database field descriptions for each mapped field.

In [ ]:
database_description = {
    'raw_store_name': 'Original store name from Excel source; stored in pickups.raw_store_name',
    'pickup_date': 'Collection date; stored in pickups.pickup_date',
    'week_number': 'Week number from legacy source; stored in pickups.week_number',
    'scg_mass_lbs': 'Raw SCG mass in pounds; stored in pickups.scg_mass_lbs',
    'net_lbs': 'Net pounds after adjustments; stored in pickups.net_lbs',
    'pickup_initiated_by': 'Collector name or identifier; stored in pickups.pickup_initiated_by',
    'master_gardener': 'Master Gardener contribution in pounds; stored in pickups.master_gardener',
    'mg_deposit_date': 'Deposit date for Master Gardener contributions; stored in pickups.mg_deposit_date',
    'cardboard_lbs': 'Cardboard weight in pounds; stored in pickups.cardboard_lbs',
    'food_waste_lbs': 'Food waste weight in pounds; stored in pickups.food_waste_lbs',
    'route': 'Collection route; stored in pickups.route',
    'miles_driven': 'Miles driven for collection; stored in pickups.miles_driven',
    'truck_odometer': 'Truck odometer reading; stored in pickups.truck_odometer',
    'notes': 'Legacy notes field; stored in pickups.notes',
    'raw_days_between_collections': 'Raw days between collections from source Excel; stored in pickups.raw_days_between_collections',
    'raw_days_since_first_collection': 'Raw days since first collection from source Excel; stored in pickups.raw_days_since_first_collection',
    'co2e_lbs': 'Calculated CO2e in pounds; derived in vw_pickup_base.co2e_lbs',
    'transportation_co2e': 'Calculated transportation CO2e; derived in vw_pickup_base.transportation_co2e',
    'scg_lbs_per_mile': 'SCG pounds per mile; derived in vw_pickup_base.scg_lbs_per_mile',
    'co2e_avoided_per_mile': 'CO2e avoided per mile; derived in vw_pickup_base.co2e_avoided_per_mile',
    'per_day_lbs': 'Average SCG pounds per day; derived in vw_pickup_base.per_day_lbs',
    'rubicon_period': 'Rubicon transition label; derived in vw_pickup_base.rubicon_period',
    'year_and_week': 'Year and ISO week string; derived in vw_pickup_base.year_and_week',
    'days_since_first_collection': 'Days since first recorded collection; derived in vw_pickup_base.days_since_first_collection',
    'days_between_collections': 'Days between collections; derived in vw_pickup_base.days_between_collections',
    'running_total': 'Cumulative net pounds; derived in vw_pickup_base.running_total',
}
catalog_df['database_field_description'] = catalog_df['target_database_field_name'].map(database_description).fillna('Definition unavailable; review mapping manually.')
catalog_df

## Export annotated field catalog to Excel

Write the assembled catalog to `field_catalog.xlsx` with the legacy field name, definition, origin label, calculated field flag, formula text, and target database field name.

In [ ]:
catalog_df.to_excel(output_catalog_path, index=False, sheet_name='Field Catalog')
print('Wrote annotated field catalog to:', output_catalog_path)